# Neo4j를 활용한 GraphRAG (Graph Retrieval-Augmented Generation)

## 1. RAG (Retrieval-Augmented Generation)란?

### 1.1 RAG의 필요성

**문제점**: 대규모 언어모델(LLM)의 한계
- ❌ **지식의 한계**: 학습 데이터 시점까지의 정보만 알고 있음
- ❌ **환각(Hallucination)**: 없는 정보를 만들어내는 경향
- ❌ **도메인 특화 지식 부족**: 특정 기업/조직의 내부 정보 모름
- ❌ **업데이트 어려움**: 새로운 정보를 반영하려면 재학습 필요

**해결책**: RAG (Retrieval-Augmented Generation)
- ✅ **최신 정보 제공**: 외부 데이터베이스에서 관련 정보 검색
- ✅ **정확성 향상**: 검색된 실제 데이터 기반 답변 생성
- ✅ **출처 제공**: 답변의 근거를 명확히 제시
- ✅ **비용 효율적**: 전체 모델 재학습 없이 정보 업데이트


### 1.2 RAG의 기본 구조

RAG는 크게 3단계로 구성됩니다:

```
┌─────────────┐
│  1. Index   │  문서를 벡터로 변환하여 DB에 저장
└──────┬──────┘
       │
       ↓
┌─────────────┐
│ 2. Retrieve │  질문과 유사한 문서를 검색
└──────┬──────┘
       │
       ↓
┌─────────────┐
│ 3. Generate │  LLM이 검색된 문서를 참고하여 답변 생성
└─────────────┘
```

**단계별 상세**:

1. **Index (색인)**: 
   - 문서를 작은 청크(chunk)로 분할
   - 임베딩 모델로 벡터화
   - 데이터베이스에 저장

2. **Retrieve (검색)**:
   - 사용자 질문을 벡터화
   - 유사도 기반으로 관련 문서 검색
   - Top-K 개의 가장 관련있는 문서 추출

3. **Generate (생성)**:
   - 검색된 문서를 컨텍스트로 제공
   - LLM이 질문 + 컨텍스트를 받아 답변 생성
   - 근거가 있는 정확한 답변 제공


## 2. VectorDB RAG vs GraphDB RAG 비교

### 2.1 Vector RAG (전통적인 RAG)

**데이터 저장 방식**: 벡터 데이터베이스 (예: Pinecone, Chroma, FAISS)

```
문서 A → [0.2, 0.8, 0.1, ...] ───┐
문서 B → [0.5, 0.3, 0.9, ...] ───┤ 벡터 DB
문서 C → [0.1, 0.7, 0.4, ...] ───┘
```

**장점** ✅:
- 의미적 유사도 검색에 강함
- 간단한 구조로 빠른 구현 가능
- 대용량 데이터 처리에 효율적
- 임베딩 기반 검색 성능 우수

**단점** ❌:
- **관계 정보 손실**: 엔티티 간 연결 관계를 파악하기 어려움
- **맥락 파악 한계**: "누가 누구와 어떤 관계인가?" 같은 질문에 약함
- **복잡한 추론 어려움**: 여러 단계의 관계 추론이 필요한 질문에 한계
- **중복 정보**: 같은 엔티티가 여러 문서에 분산되어 저장

**적합한 사용 사례**:
- 단순 키워드/의미 검색
- FAQ 시스템
- 문서 유사도 검색


### 2.2 Graph RAG (그래프 기반 RAG)

**데이터 저장 방식**: 그래프 데이터베이스 (예: Neo4j, Amazon Neptune)

```
    (기자A)──[WORKS_FOR]──→(언론사X)
       │                         ↑
       │                         │
   [WRITTEN_BY]            [PUBLISHED_BY]
       │                         │
       ↓                         │
    (뉴스1)──[BELONGS_TO]──→(카테고리)
       │
   [PUBLISHED_IN]
       │
       ↓
   (2025-11)
```

**장점** ✅:
- **관계 보존**: 엔티티 간의 연결 관계를 명시적으로 저장
- **복잡한 질의**: "이 기자가 작성한 경제 뉴스는?" 같은 복잡한 질문 처리
- **맥락 이해**: 그래프 경로를 따라가며 풍부한 맥락 제공
- **추론 가능**: 여러 홉(hop)을 거쳐 관계 추론
- **중복 제거**: 같은 엔티티는 하나의 노드로 통합

**단점** ❌:
- 초기 그래프 구축이 복잡함
- 벡터 검색보다 구현이 어려움
- 그래프 설계가 성능에 큰 영향

**적합한 사용 사례**:
- 지식 그래프 기반 QA
- 복잡한 관계 분석 (소셜 네트워크, 추천 시스템)
- 엔티티 중심의 검색 (특정 인물, 기업, 제품)
- 다단계 추론이 필요한 질문


### 2.3 비교표

| 항목 | Vector RAG | Graph RAG |
|------|-----------|-----------|
| **데이터 구조** | 벡터 (Embeddings) | 노드 + 관계 (Graph) |
| **검색 방식** | 코사인 유사도 | Cypher 쿼리 (경로 탐색) |
| **관계 표현** | ❌ 어려움 | ✅ 명시적 |
| **구현 난이도** | ⭐⭐ 쉬움 | ⭐⭐⭐⭐ 어려움 |
| **복잡한 질의** | ❌ 한계 있음 | ✅ 강력함 |
| **확장성** | ✅ 우수 | ⚠️ 그래프 크기에 따라 다름 |
| **추론 능력** | ❌ 제한적 | ✅ 강력함 |
| **유지보수** | ⭐⭐ 쉬움 | ⭐⭐⭐ 보통 |


## 3. Neo4j를 활용한 GraphRAG 실습 준비작업



### 3.1 API Key 설정
- [OpenAI API Key 발급](https://platform.openai.com/api-keys)


In [1]:
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv

# 환경변수 로드
load_dotenv()


True

### 3.2 Neo4j 연결 클래스

In [2]:
class Singleton(type):
    _instances = {}
    
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super(Singleton, cls).__call__(*args, **kwargs)
        return cls._instances[cls]

In [3]:
# Neo4j 연결 클래스 (Singleton 패턴)
from neo4j import GraphDatabase

class Neo4jConnection(metaclass=Singleton):
    """Neo4j 데이터베이스 연결 및 작업을 위한 클래스"""
    
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        print("Neo4j 연결 성공!")
    
    def close(self):
        if self.driver is not None:
            self.driver.close()
            print("Neo4j 연결 종료")
    
    def execute_query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return [record for record in result]


### 3.3 Neo4j 연결 및 데이터 확인 

In [10]:
# Neo4j 연결
neo4j_conn = Neo4jConnection(
    uri="bolt://localhost:7687",
    user="neo4j",
    password="test1234"
)


In [11]:
# 저장된 데이터 확인
query = """
MATCH (n)
RETURN labels(n)[0] as NodeType, count(n) as Count
ORDER BY Count DESC
"""

results = neo4j_conn.execute_query(query)
print("Neo4j 데이터 현황:\n")
for record in results:
    print(f"  {record['NodeType']:<15} : {record['Count']:>3}개")


Neo4j 데이터 현황:

  Reporter        :  20개
  News            :  20개
  Publisher       :  15개
  Category        :   4개
  YearMonth       :   1개


## 4. GraphRAG 시스템 구축

### 4.1 질문 유형별 Cypher 쿼리 템플릿

GraphRAG의 핵심은 **자연어 질문을 Cypher 쿼리로 변환**하는 것입니다!


In [12]:
from enum import Enum

class CypherQueryTemplates(Enum):
    """다양한 질문 유형을 Enum으로 관리하는 Cypher 템플릿"""

    NEWS_BY_CATEGORY = "news_by_category"
    NEWS_BY_PUBLISHER = "news_by_publisher"
    NEWS_BY_REPORTER = "news_by_reporter"
    NEWS_BY_PUBLISHER_AND_CATEGORY = "news_by_publisher_and_category"
    REPORTER_STATISTICS = "reporter_statistics"
    SEARCH_BY_KEYWORD = "search_by_keyword"

    def build(self, **kwargs):
        """템플릿 유형에 따라 Cypher 쿼리 생성"""
        if self is CypherQueryTemplates.NEWS_BY_CATEGORY:
            category_name = kwargs["category_name"]
            return f"""
            MATCH (n:News)-[:BELONGS_TO]->(c:Category {{name: "{category_name}"}})
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   p.name as 언론사, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT 5
            """

        if self is CypherQueryTemplates.NEWS_BY_PUBLISHER:
            publisher_name = kwargs["publisher_name"]
            return f"""
            MATCH (n:News)-[:PUBLISHED_BY]->(p:Publisher {{name: "{publisher_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT 5
            """

        if self is CypherQueryTemplates.NEWS_BY_REPORTER:
            reporter_name = kwargs["reporter_name"]
            return f"""
            MATCH (n:News)-[:WRITTEN_BY]->(r:Reporter {{name: "{reporter_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, p.name as 언론사, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT 5
            """

        if self is CypherQueryTemplates.NEWS_BY_PUBLISHER_AND_CATEGORY:
            publisher_name = kwargs["publisher_name"]
            category_name = kwargs["category_name"]
            return f"""
            MATCH (n:News)-[:PUBLISHED_BY]->(p:Publisher {{name: "{publisher_name}"}})
            MATCH (n)-[:BELONGS_TO]->(c:Category {{name: "{category_name}"}})
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT 5
            """

        if self is CypherQueryTemplates.REPORTER_STATISTICS:
            return """
            MATCH (r:Reporter)-[:WRITTEN_BY]-(n:News)
            MATCH (r)-[:WORKS_FOR]->(p:Publisher)
            WITH r.name as 기자명, p.name as 소속언론사, count(n) as 작성기사수
            RETURN 기자명, 소속언론사, 작성기사수
            ORDER BY 작성기사수 DESC
            LIMIT 10
            """

        if self is CypherQueryTemplates.SEARCH_BY_KEYWORD:
            keyword = kwargs["keyword"]
            return f"""
            MATCH (n:News)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            WHERE n.title CONTAINS "{keyword}" OR n.content CONTAINS "{keyword}"
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, p.name as 언론사, r.name as 기자,
                   n.link as 뉴스링크
            LIMIT 5
            """

        raise ValueError(f"지원하지 않는 템플릿: {self}")


#### 테스트 

##### 카테고리로 뉴스 검색

In [32]:
# 카테고리로 뉴스 검색
query = CypherQueryTemplates.NEWS_BY_CATEGORY.build(category_name="경제")
print(query)


            MATCH (n:News)-[:BELONGS_TO]->(c:Category {name: "경제"})
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            RETURN n.title as 제목, n.content as 내용, 
                   p.name as 언론사, r.name as 기자, n.publishDate as 발행일,
                   n.link as 뉴스링크
            LIMIT 5
            


In [33]:
results = neo4j_conn.execute_query(query)

In [34]:
results

[<Record 제목='1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑' 내용="10월 세수 전년比 2.8조 늘어…법인세 7000억·소득세 9000억↑ 진도율 88.9%, 전년比 1.7%p↑…5년 평균과 유사 ⓒ News1 윤주희 디자이너 (세종=뉴스1) 임용우 기자 = 올해 1~10월 국세 수입이 지난해 같은 기간보다 37조 1000억 원 늘어난 것으로 집계됐다. 지난해와 올해 상반기 기업 실적 개선, 성과급 지급 확대 등이 이어지며 법인세와 소득세가 크게 증가한 영향이다. 기획재정부가 28일 발표한 '2025년 10월 국세수입 현황'에 따르면 10월 국세 수입은 41조 1000억 원으로 전년 동월(38조 3000억 원)보다 2조 8000억 원 증가했다. 법인세는 상반기 기업 실적 개선에 따른 중소기업 중간예납 분납분 증가, 이자·배당 원천소득 증가 등으로 7000억 원 늘었다. 소득세는 근로자 수 증가와 총급여 지급액 확대에 따른 근로소득세 증가로 9000억 원 증가했다. 부가가치세는 올해 2기 예정신고분 납부 증가와 환율 상승 영향 등으로 7000억 원 증" 언론사='뉴스1' 기자='임용우 기자' 발행일='2025-11-28 11:00:00' 뉴스링크='https://n.news.naver.com/mnews/article/421/0008630778'>,
 <Record 제목='지난 자율주행차 사고 47건…교통안전공단 통계 최초 공개' 내용="주행속도 시속 30㎞ 이하 저속 구간서 주로 발생 내년엔 '차량거동' 정보까지 확대 공개 자율주행자동차 사고통계 공개 누리집 모습.(한국교통안전공단 제공)뉴스1ⓒ news1 (서울=뉴스1) 김동규 기자 = 한국교통안전공단 자동차안전연구원(TS)이 정부와 함께 국내 자율주행차 사고통계를 처음으로 공개한다. 2022년 이후 사고가 매년 증가하는 가운데 자율주행모드 운행 중 사고가 전체의 35%에 이르는 것으로 나타나, 안전성 확보를 위한 체계적 데이터 공개가 본격화됐다는 평가다. TS 자

In [36]:
for record in results:
    print(f"제목: {record['제목']}")
    print(f"내용: {record['내용'][:50]}...")
    print(f"언론사: {record['언론사']}")
    print(f"뉴스링크: {record['뉴스링크']}")
    break 

제목: 1~10월 국세수입 37.1조 증가한 330.7조…기업 실적 개선에 법인·소득세↑
내용: 10월 세수 전년比 2.8조 늘어…법인세 7000억·소득세 9000억↑ 진도율 88.9%,...
언론사: 뉴스1
뉴스링크: https://n.news.naver.com/mnews/article/421/0008630778


##### 기자 통계 조회

In [37]:
# 기자 통계 조회
query = CypherQueryTemplates.REPORTER_STATISTICS.build()
print(query)


            MATCH (r:Reporter)-[:WRITTEN_BY]-(n:News)
            MATCH (r)-[:WORKS_FOR]->(p:Publisher)
            WITH r.name as 기자명, p.name as 소속언론사, count(n) as 작성기사수
            RETURN 기자명, 소속언론사, 작성기사수
            ORDER BY 작성기사수 DESC
            LIMIT 10
            


In [38]:
results = neo4j_conn.execute_query(query)

In [39]:
results

[<Record 기자명='장훈경 기자' 소속언론사='SBS' 작성기사수=1>,
 <Record 기자명='정남구 기자' 소속언론사='한겨레' 작성기사수=1>,
 <Record 기자명='신기림 기자' 소속언론사='뉴스1' 작성기사수=1>,
 <Record 기자명='김성식 기자' 소속언론사='뉴스1' 작성기사수=1>,
 <Record 기자명='임용우 기자' 소속언론사='뉴스1' 작성기사수=1>,
 <Record 기자명='김동규 기자' 소속언론사='뉴스1' 작성기사수=1>,
 <Record 기자명='김종윤 기자' 소속언론사='SBS Biz' 작성기사수=1>,
 <Record 기자명='김지헌 기자' 소속언론사='헤럴드경제' 작성기사수=1>,
 <Record 기자명='함영훈 기자' 소속언론사='헤럴드경제' 작성기사수=1>,
 <Record 기자명='이광수 기자' 소속언론사='국민일보' 작성기사수=1>]

In [40]:
for record in results:
    print(f"기자명: {record['기자명']}")
    print(f"소속언론사: {record['소속언론사']}")
    print(f"작성기사수: {record['작성기사수']}")
    break 

기자명: 장훈경 기자
소속언론사: SBS
작성기사수: 1


##### 키워드 검색

In [41]:
# 키워드 검색
query = CypherQueryTemplates.SEARCH_BY_KEYWORD.build(keyword="자율주행")
print(query)


            MATCH (n:News)-[:BELONGS_TO]->(c:Category)
            MATCH (n)-[:PUBLISHED_BY]->(p:Publisher)
            MATCH (n)-[:WRITTEN_BY]->(r:Reporter)
            WHERE n.title CONTAINS "자율주행" OR n.content CONTAINS "자율주행"
            RETURN n.title as 제목, n.content as 내용, 
                   c.name as 카테고리, p.name as 언론사, r.name as 기자,
                   n.link as 뉴스링크
            LIMIT 5
            


In [42]:
results = neo4j_conn.execute_query(query)

In [43]:
results

[<Record 제목='지난 자율주행차 사고 47건…교통안전공단 통계 최초 공개' 내용="주행속도 시속 30㎞ 이하 저속 구간서 주로 발생 내년엔 '차량거동' 정보까지 확대 공개 자율주행자동차 사고통계 공개 누리집 모습.(한국교통안전공단 제공)뉴스1ⓒ news1 (서울=뉴스1) 김동규 기자 = 한국교통안전공단 자동차안전연구원(TS)이 정부와 함께 국내 자율주행차 사고통계를 처음으로 공개한다. 2022년 이후 사고가 매년 증가하는 가운데 자율주행모드 운행 중 사고가 전체의 35%에 이르는 것으로 나타나, 안전성 확보를 위한 체계적 데이터 공개가 본격화됐다는 평가다. TS 자동차안전연구원은 28일 국토교통부와 함께 자율주행차 사고 정보를 정례적으로 공개한다고 밝혔다. 제한적으로 관리되던 사고 데이터를 투명하게 공개하는 것은 이번이 처음으로, 자율주행차 상용화를 앞두고 국민 신뢰도를 높이기 위한 조치다. 사고통계는 '자율주행자동차 사고조사위원회' 누리집을 통해 확인할 수 있다. TS는 제작사로부터 제출받는 자율주행차 교통사고 신고서를 바탕으로 △연도 △운행모드 △시간대 △날씨" 카테고리='경제' 언론사='뉴스1' 기자='김동규 기자' 뉴스링크='https://n.news.naver.com/mnews/article/421/0008631187'>]

In [44]:
for record in results:
    print(f"제목: {record['제목']}")
    print(f"내용: {record['내용'][:50]}...")
    print(f"언론사: {record['언론사']}")
    print(f"뉴스링크: {record['뉴스링크']}")
    break 

제목: 지난 자율주행차 사고 47건…교통안전공단 통계 최초 공개
내용: 주행속도 시속 30㎞ 이하 저속 구간서 주로 발생 내년엔 '차량거동' 정보까지 확대 공개 ...
언론사: 뉴스1
뉴스링크: https://n.news.naver.com/mnews/article/421/0008631187
